In [1]:
import pandas as pd
import numpy as np
import sys
PROJECT_ROOT = '/home/claud/pontia/PJ'
sys.path.insert(0,PROJECT_ROOT)
from io import StringIO
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
nomenclator_df= pd.read_csv(f'{PROJECT_ROOT}/data/raw/nomenclator.csv')
streets_list = list(nomenclator_df['nom_oficial'])

In [63]:
with open(f'{PROJECT_ROOT}/data/raw/carrers_separat_tipus.csv') as f:
    nomencaltor_s = f.read().replace('\"','')


In [5]:
street_types = [e.split(',')[1] for e in streets_list]
words_list = set()
for e in street_types:
    words_list.update(e.split())
words_list = list(words_list)

In [6]:
words_list

['carrer',
 'moll',
 "n'",
 'plaça',
 'pas',
 'jardí',
 'parc',
 'espigó',
 'carreró',
 'avinguda',
 'les',
 'escales',
 'rotonda',
 'camí',
 'mirador',
 'placeta',
 'passatge',
 'jardins',
 'via',
 'passeig',
 'dels',
 'carretera',
 'de',
 "d'",
 "l'"]

In [7]:
words_list_red_es =[
 'escaleras',
 'vía',
 'parque',
 'espigón',
 'pasaje',
 'jardín',
 'calle',
 'avenida',
 'camino',
 'muelle',
 'paso',
 'jardines',
 'callejuela',
 'placita',
 'plaza',
 'paseo',
]

In [8]:
words_list_red_cat = [
 'escales',
 'via',
 'parc',
 'espigó',
 'passatge',
 'jardí',
 'carrer',
 'avinguda',
 'camí',
 'moll',
 'pas',
 'jardins',
 'carreró',
 'placeta',
 'plaça',
 'passeig',
]

In [9]:
es2cat = {es:cat for es, cat in zip(words_list_red_es, words_list_red_cat)}
cat2es = {cat:es for es, cat in zip(words_list_red_es, words_list_red_cat)}

In [64]:
streets_list_s = '\n'.join(streets_list)
streets_list_s_es = streets_list_s
for catw in cat2es.keys():
    streets_list_s_es = streets_list_s_es.replace(catw, cat2es[catw])
streets_list_es = streets_list_s_es.split('\n')[1:]

In [3]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 1), # por jemplo Carrer es siempre relevante como parte de un digrama. Haty nombre largos. Se ha probado que con este setting se ontienen buenos resultados
    token_pattern=r"(?u)\b\w+\b", # para que tenga en cuenta las palabras de una sola letra, como F en Carrer F
    use_idf=False, # Aunaque Carrer o Plaça son términos muy comunes, son totalmente relevantes, para distinguir Carrer de l'Àngel de Plça de l'Àngel, por ejemplo
)
""" tfidf_vectorizer_es = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 1),
) """
tfidf_matrix = tfidf_vectorizer.fit_transform(streets_list)
""" tfidf_matrix_es = tfidf_vectorizer_es.fit_transform(streets_list_es)
 """

' tfidf_matrix_es = tfidf_vectorizer_es.fit_transform(streets_list_es)\n '

In [4]:
query = tfidf_vectorizer.transform(["Carrer de Londres 20"])
#dp = np.dot(tfidf_matrix, query.T)
dp = (tfidf_matrix @ query.T).toarray().ravel()
ranking = sorted(zip(streets_list, dp), key=lambda x: x[1], reverse=True)
res = []
for doc, score in ranking:
    if score > 0:
        res.append((doc, score))
res


[('Carrer de Londres', np.float64(0.8660254037844388)),
 ('Carrer de Casp', np.float64(0.5773502691896258)),
 ('Carrer de Raset', np.float64(0.5773502691896258)),
 ('Carrer de Pelfort', np.float64(0.5773502691896258)),
 ('Carrer de Casteràs', np.float64(0.5773502691896258)),
 ('Carrer de Rogent', np.float64(0.5773502691896258)),
 ('Carrer de Xifré', np.float64(0.5773502691896258)),
 ('Carrer de Sorolla', np.float64(0.5773502691896258)),
 ('Carrer de Palència', np.float64(0.5773502691896258)),
 ('Carrer de Bascònia', np.float64(0.5773502691896258)),
 ('Carrer de Margarit', np.float64(0.5773502691896258)),
 ('Carrer de Calaf', np.float64(0.5773502691896258)),
 ('Carrer de Mercedes', np.float64(0.5773502691896258)),
 ('Carrer de Berna', np.float64(0.5773502691896258)),
 ('Carrer de Logronyo', np.float64(0.5773502691896258)),
 ('Carrer de Jorba', np.float64(0.5773502691896258)),
 ('Carrer de Coïmbra', np.float64(0.5773502691896258)),
 ('Carrer de Gelabert', np.float64(0.5773502691896258)),

In [155]:
dp.toarray().ravel()


array([0., 0., 0., ..., 0., 0., 0.], shape=(4770,))

In [98]:
zip(streets_list, dp)

In [83]:
dp.data

array([0.05578867, 0.05129928, 0.05781517, 0.04362734, 0.06394088,
       0.06614506, 0.05180644, 0.06394088, 0.06394088, 0.0427013 ,
       0.06042386, 0.06151695, 0.04939747, 0.05931072, 0.06662258,
       0.06394088, 0.05129928, 0.04558772, 0.03761538, 0.06394088,
       0.05104583, 0.07171695, 0.04632242, 0.06394088, 0.06435378,
       0.06123739, 0.55641878, 0.045738  , 0.06267161, 0.04954754,
       0.05344266, 0.06394088, 0.06394088, 0.04845392, 0.05161331,
       0.06734694, 0.04633572, 0.07070312, 0.04201917, 0.04678046,
       0.04808026, 0.04264836, 0.06524043, 0.05415679, 0.03850136,
       0.04072662, 0.06394088, 0.06865412, 0.06946704, 0.04902544,
       0.06394088, 0.46691823, 0.06394088, 0.03919354, 0.0601762 ,
       0.07090262, 0.04764736, 0.06394088, 0.06394088, 0.06394088,
       0.06865412, 0.04605812, 0.04486203, 0.04880803, 0.05913539,
       0.05179308, 0.05510467, 0.21626516, 0.04824996, 0.05496953,
       0.06394088, 0.06394088, 0.04323093, 0.06662258, 0.06042

In [ ]:
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print(f"Documents: {tfidf_matrix.shape[0]}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Non-zero weights: {tfidf_matrix.nnz}")

# Inspect the strongest terms for the first street title.
first_row_weights = tfidf_matrix[0].toarray().ravel()
pd.Series(first_row_weights, index=tfidf_terms).nlargest(10)

In [42]:
with open(f'{PROJECT_ROOT}/data/raw/nomenclator.csv', 'r') as f:
    nomenclator_df = pd.read_csv(StringIO(f.read().replace('"','')), sep=',')

In [8]:
nomenclator_df = pd.read_csv(f'{PROJECT_ROOT}/data/raw/nomenclator.csv')

In [20]:
direction_tyes = list(set([r[1] for r in nomenclator_df['title'].str.split(',')]))

In [24]:
direction_types_es = [' camino', ' parque de les', ' plaza de les', ' carretera', " parque d'", ' calle dels', " espigón d'", ' plaza dels', ' calle', ' muelle', ' camino dels', ' pasaje', " avenida de l'", " paseo de l'", " paso de l'", " calle d'", " pasaje d'", " calle de l'", ' carreró de les', ' pasaje dels', ' via', " jardines de l'", " mirador de l'", ' jardín dels', " avenida d'", " plaza d'", ' parque dels', ' plaza', " jardines d'", ' calle de les', " plaza de l'", " escaleras d'", " muelle d'", " rotonda de l'", ' carretera de les', " paso d'", " jardín d'", ' camino de les', " paseo d'", " placeta d'", " pasaje de l'", " calle de n'"]

In [23]:
print(list(set([r[1] for r in nomenclator_df['title'].str.split(',')])))

[' camí', ' parc de les', ' plaça de les', ' carretera', " parc d'", ' carrer dels', " espigó d'", ' plaça dels', ' carrer', ' moll', ' camí dels', ' passatge', " avinguda de l'", " passeig de l'", " pas de l'", " carrer d'", " passatge d'", " carrer de l'", ' carreró de les', ' passatge dels', ' via', " jardins de l'", " mirador de l'", ' jardí dels', " avinguda d'", " plaça d'", ' parc dels', ' plaça', " jardins d'", ' carrer de les', " plaça de l'", " escales d'", " moll d'", " rotonda de l'", ' carretera de les', " pas d'", " jardí d'", ' camí de les', " passeig d'", " placeta d'", " passatge de l'", " carrer de n'"]


In [26]:
streets_dict = {}

In [27]:
streets_dict['street_name'] = [r[0] for r in nomenclator_df['title'].str.split(',')]

In [28]:
streets_dict['street_type_cat'] = [r[1] for r in nomenclator_df['title'].str.split(',')]

In [29]:
streets_dict['street_nomenclator'] = [f'{st} {sn}' for st, sn in zip(streets_dict['street_type_cat'], streets_dict['street_name'])]

In [30]:
streets_dict['street_nomenclator']

[' carrer A Zona Franca',
 " carrer de l' Abadessa Olzet",
 ' carrer dels Abaixadors',
 " plaça de l' Abat Escarré",
 " carrer de l' Abat Odó",
 " plaça de l' Abat Oliba",
 " carrer de l' Abat Safont",
 " carrer de l' Abat Samsó",
 " carrer d' Abd el-Kader",
 " carrer d' Abdó Terradas",
 " carrer d' Abella",
 ' carrer de les Acàcies',
 " plaça de l' Acadèmia",
 " carrer de l' Acer",
 ' carrer de les Açores',
 " carrer de l' Actor Morano",
 " carrer de l' Actriu Tubau",
 " jardins d' Ada Byron",
 " carrer d' Adolf Florensa",
 " plaça d' Adolf Marsillach",
 ' moll Adossat',
 " avinguda d' Adrià Gual",
 " carrer d' Adrià Margarit",
 " plaça d' Adrià",
 ' carrer dels Afores',
 " carrer d' Àgata Badia Puig-Rodon",
 " carrer de n' Aglà",
 " carrer d' Agramunt",
 " carrer de l' Agregació",
 " passeig de l' Agrícola",
 " carrer de l' Agricultura",
 ' carrer dels Agudells',
 ' carrer de les Agudes',
 " carrer d' Aguilar",
 " carrer d' Agullana",
 ' carrer dels Agullers',
 " jardins d' Agustí Ce

In [ ]:
streets_dict['street_nomenclator']

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "data/raw/nomenclator.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate data/raw/nomenclator.csv from the current directory.")

data_path = project_root / "data/raw/nomenclator.csv"
nomenclator_df = pd.read_csv(data_path)
street_titles = nomenclator_df["title"].fillna("").astype(str).str.strip()

# Each title is a document; word and two-word phrase features capture street names and types.
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
)
tfidf_matrix = tfidf_vectorizer.fit_transform(street_titles)
tfidf_terms = tfidf_vectorizer.get_feature_names_out()

print(f"Documents: {tfidf_matrix.shape[0]}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Non-zero weights: {tfidf_matrix.nnz}")

# Inspect the strongest terms for the first street title.
first_row_weights = tfidf_matrix[0].toarray().ravel()
pd.Series(first_row_weights, index=tfidf_terms).nlargest(10)